In [1]:
print("Hiii")

Hiii


In [2]:
%pwd

'c:\\Users\\lenovo\\End-to-End-Medical-Chatbot\\research'

In [2]:
import os
from pathlib import Path


In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\lenovo\anaconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#Extract Data from PDF file
def load_pdf_file(data):
    data_path = Path(data)
    candidate_paths = [
        data_path,
        Path.cwd() / data_path,
        Path.cwd().parent / data_path,
    ]
    resolved_path = next((path.resolve() for path in candidate_paths if path.exists()), None)

    if resolved_path is None:
        checked_paths = ", ".join(str(path.resolve()) for path in candidate_paths)
        raise FileNotFoundError(
            f"PDF directory not found. Create the folder and add PDF files there. Checked: {checked_paths}"
        )

    loader= DirectoryLoader(str(resolved_path), glob="*.pdf", loader_cls=PyPDFLoader)
    documents=loader.load()
    return documents


In [5]:
extracted_data=load_pdf_file(data='Data/')

In [6]:
#split and chunk the data
def text_splitter(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [7]:
text_chunks=text_splitter(extracted_data)
print("length of text chunks:",len(text_chunks))

length of text chunks: 5860


In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [9]:
#download embeddings from huggingface
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [10]:
embeddings = download_hugging_face_embeddings()


C:\Users\lenovo\AppData\Local\Temp\ipykernel_20732\3183104011.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3606.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
query_result= embeddings.embed_query("What is the capital of France?")
print("len query result:", len(query_result))

len query result: 384


In [12]:
from dotenv import load_dotenv

load_dotenv()

from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [18]:


index_name = "medicalbot"

pc.create_index(
name=index_name,
dimension=384,
metric="cosine",
spec=ServerlessSpec(
cloud="aws",
region="us-east-1"
)
)

PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': 'f5720a774ffce39fdd209192d00c1df9', 'date': 'Tue, 31 Mar 2026 08:18:31 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [45]:
import os
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

In [46]:
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# Create OpenAI client pointing to OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

In [20]:
#embed each chunk and upsert to pinecone index
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(documents=text_chunks, index_name=index_name, embedding=embeddings)

In [19]:
#load existing index

from langchain_pinecone import PineconeVectorStore
#embed each chunk and upsert to pinecone index
#connection
docsearch= PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embeddings)


In [20]:
docsearch

In [21]:
retriever = docsearch.as_retriever(search_type= "similarity" , search_kwargs={"k": 3})

In [22]:
retrieved_docs= retriever.invoke("What is the capital of France?")

In [23]:
retrieved_docs

[Document(id='4d5bdba6-c010-4dcc-83e5-22f020572301', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 350.0, 'page_label': '351', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'C:\\Users\\lenovo\\End-to-End-Medical-Chatbot\\Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='benzene and certain insecticides)\n• infection with certain viruses (especially those causing\nviral hepatitis, as well as Epstein-Barr virus, par-\nvovirus, and HIV , the virus which can cause AIDS)\n• pregnancy\n• certain other disorders, including a disease called parox-\nysmal nocturnal hemoglobinuria, an autoimmune reac-\ntion called graft-vs-host disease (which occurs when the\nbody’s immune system attacks and destroys the body’s\nown cells), and certain connective tissue diseases'),
 Document(id='020ea965-695f-4c25-ae0e-f52f615edbe5', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2

In [41]:
#Integrating LLM

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY, model="openai/gpt-3.5-turbo", temperature=0.4, max_tokens=500)

In [25]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


In [26]:
system_prompt = ("You are a helpful assistant for answering questions based on the retrieved documents. "
"Use the following retrieved documents to answer the question. If you don't know the answer, say you don't know. Use three sentences maximum to answer the question. Be concise and to the point. \n\n"
"{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])


In [42]:
question_answering_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answering_chain)

In [48]:
# Use ainvoke instead of invoke
response = await rag_chain.ainvoke({"input": "What is acne?"})
print(response["answer"])

Acne is a skin disorder that occurs when the sebaceous glands become inflamed. It is commonly known as acne vulgaris and can affect various parts of the body, including the face. It is characterized by the presence of pimples, blackheads, and cysts on the skin.


In [50]:
# Use ainvoke instead of invoke
response = await rag_chain.ainvoke({"input": "What is gigantism?"})
print(response["answer"])

Gigantism is a disorder caused by the excessive release of growth hormone during childhood and adolescence, leading to abnormal growth and excessive height. It results in an individual being significantly taller than average due to the overgrowth of bones and tissues. Gigantism is a rare condition that can have serious health implications if left untreated.
